In [1]:
import os
import pandas as pd
import numpy as np
import re
import requests

In [2]:
# Define user
user = os.getlogin()

# Set file paths
path_sp  = os.path.join('C:\\Users', user, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_git = os.path.join('C:\\Users', user, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config = os.path.join(path_git, 'Python Code', 'DOF', 'config')
print(path_sp)

C:\Users\jfontes\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents


In [3]:
# ## User defined functions
# exec(open(os.path.join(path_config, 'Functions.py')).read())

In [4]:
## Import Variable Mapping
df_inputs = pd.read_excel(os.path.join(path_config, 'DOF Configuration File.xlsx'), sheet_name = 'Inputs', usecols = 'A:E')

# Organize inputs into run
indicator_name = df_inputs['indicator_name'].values[0]
report_theme   = df_inputs['report_theme'  ].values[0]
sp_folder_out  = df_inputs['sp_folder'     ].values[0]
counties       = df_inputs['counties'].dropna().values

print(indicator_name)
print(report_theme  )
print(sp_folder_out )
print(counties      )

Pop_1
Vibrant and Inclusive Places
People and Community\\Pop and Demographics
['El Dorado' 'Placer' 'Sacramento' 'Sutter' 'Yolo' 'Yuba']


***

## Pop_1

***

#### Counties

In [5]:
## Indicator Pop_1 Datasets

if indicator_name == 'Pop_1':
    
    ## Outline
    # Set fake user agent to avoid 403 error
    # Set URL of table, starting here https://dof.ca.gov/forecasting/demographics/
    # Request import of excel workbook with URL
    # Convert request content to pandas dataframe
    # Drop missings, remove state totals
    # Reshape data to long format, clean date field, reshape back to wide
    # Clean column names
    # Repeat for data from 2000-2010, 2010-2020, 2020-2024
    # Stack all data together
    # Subset to counties as needed
    
    
    # i got lucky with finding this user agent on stackoverflow, not sure why it works
    headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/114.0'} 
    
    
    ## Import data
    # 2000-2010
    url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E4_2000-2010_Report_Final_EOC_000.xlsx"
    request = requests.get(url, headers = headers)
    request = request.content
    df_2000w = pd.read_excel(request, sheet_name = 1, skiprows = 3, engine='openpyxl')
    df_2000w = df_2000w.dropna()
    df_2000w = df_2000w[df_2000w['COUNTY'] != 'State Total']
    df_2000 = pd.melt(df_2000w
                        , id_vars = ['COUNTY']
                        , var_name = 'Date'
                        , value_name = 'Total'
                     )
    df_2000['Date'] = pd.to_datetime(df_2000['Date'])
    df_2000w = df_2000.pivot_table(index = ['COUNTY']
                                       , columns = 'Date'
                                       , values = 'Total').reset_index()
    df_2000w.columns = [re.sub(" 00:00:00", "", str(col)) for col in df_2000w.columns]
    
    # 2010-2020
    url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2010-2020-Internet-Version.xlsx"
    request = requests.get(url, headers = headers)
    request = request.content
    df_2010w = pd.read_excel(request, sheet_name = 1, skiprows = 1, engine='openpyxl')
    df_2010w = df_2010w[df_2010w['COUNTY'] != 'State Total']
    df_2010w = df_2010w.dropna()
    df_2010 = pd.melt(df_2010w
                        , id_vars = ['COUNTY']
                        , var_name = 'Date'
                        , value_name = 'Total'
                     )
    df_2010['Date'] = pd.to_datetime(df_2010['Date'])
    df_2010w = df_2010.pivot_table(index = ['COUNTY']
                                       , columns = 'Date'
                                       , values = 'Total').reset_index()
    df_2010w.columns = [re.sub(" 00:00:00", "", str(col)) for col in df_2010w.columns]
    
    # 2020-2024
    url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2024_InternetVersion.xlsx"
    request = requests.get(url, headers = headers)
    request = request.content
    df_2020w = pd.read_excel(request, sheet_name = 1, skiprows = 2, engine='openpyxl')
    df_2020w = df_2020w[df_2020w['COUNTY'] != 'State Total']
    df_2020w = df_2020w.dropna()
    df_2020 = pd.melt(df_2020w
                        , id_vars = ['COUNTY']
                        , var_name = 'Date'
                        , value_name = 'Total'
                     )
    df_2020['Date'] = pd.to_datetime(df_2020['Date'])
    df_2020w = df_2020.pivot_table(index = ['COUNTY']
                                       , columns = 'Date'
                                       , values = 'Total').reset_index()
    df_2020w.columns = [re.sub(" 00:00:00", "", str(col)) for col in df_2020w.columns]
    
    ## Combine data
    
    # wide
    df_dof2 = df_2000w.merge(df_2010w, on = 'COUNTY', how = 'left')
    df_dof2 = df_dof2 .merge(df_2020w, on = 'COUNTY', how = 'left')
    
    # long
    df_dof = pd.concat([df_2000, df_2010, df_2020])
    df_dof = df_dof.sort_values(['COUNTY', 'Date'])

    # Clean COUNTY field
    df_dof ['COUNTY'] = df_dof ['COUNTY'].apply(lambda s : re.sub('[\s+]', ' ', s.strip()))
    df_dof2['COUNTY'] = df_dof2['COUNTY'].apply(lambda s : re.sub('[\s+]', ' ', s.strip()))

    ## Subset to counties as needed
    df_dof  = df_dof [df_dof ['COUNTY'].isin(counties)]
    df_dof2 = df_dof2[df_dof2['COUNTY'].isin(counties)]

print('Success!  DOF data imported correctly probably')

Success!  DOF data imported correctly probably


In [6]:
df_dof.head()

,COUNTY,Date,Total
8,El Dorado,2000-04-01,156299.0
66,El Dorado,2001-01-01,160079.0
124,El Dorado,2002-01-01,163300.0
182,El Dorado,2003-01-01,166195.0
240,El Dorado,2004-01-01,168984.0


In [7]:
df_dof2.head()

,COUNTY,2000-04-01,2001-01-01,2002-01-01,2003-01-01,2004-01-01,2005-01-01,2006-01-01,2007-01-01,2008-01-01,...,2016-01-01,2017-01-01,2018-01-01,2019-01-01,2020-01-01,2020-04-01,2021-01-01,2022-01-01,2023-01-01,2024-01-01
8,El Dorado,156299.0,160079.0,163300.0,166195.0,168984.0,171739.0,174218.0,176226.0,177897.0,...,183586,184993,187940,189691,193519,191185,190737,189294,188067,188583
30,Placer,248399.0,258293.0,270845.0,283703.0,296712.0,307710.0,317437.0,325985.0,333805.0,...,376307,383258,388872,395345,399015,404739,406443,407539,410085,412844
33,Sacramento,1223499.0,1248072.0,1279588.0,1307189.0,1331910.0,1350523.0,1365214.0,1380172.0,1394510.0,...,1495620,1511390,1525099,1538054,1553157,1585055,1580120,1572254,1576639,1578938
50,Sutter,78930.0,79722.0,81086.0,83018.0,85097.0,87097.0,89364.0,91563.0,92983.0,...,96823,98530,100514,102681,101339,99633,99089,99153,98248,100110
56,Yolo,168660.0,172450.0,177180.0,180798.0,184015.0,186530.0,189078.0,192826.0,196219.0,...,214884,217805,219651,220330,221276,216403,214911,220176,220454,221666


#### Jurisdictions

In [49]:
## Indicator Pop_1 Datasets

if indicator_name == 'Pop_1':
    
    ## Outline
    # Set fake user agent to avoid 403 error
    # Set URL of table, starting here https://dof.ca.gov/forecasting/demographics/
    # Request import of excel workbook with URL
    # Convert request content to pandas dataframe
    # Drop missings, remove state totals
    # Reshape data to long format, clean date field, reshape back to wide
    # Clean column names
    # Repeat for data from 2000-2010, 2010-2020, 2020-2024
    # Stack all data together
    # Subset to counties as needed
    
    
    # i got lucky with finding this user agent on stackoverflow, not sure why it works
    headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/114.0'} 
    
    
    ## Import data
    # 2000-2010
    url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E4_2000-2010_Report_Final_EOC_000.xlsx"
    request = requests.get(url, headers = headers)
    request = request.content
    df_2000w = pd.read_excel(request, sheet_name = 2, skiprows = 3, engine='openpyxl')
    df_2000w = df_2000w.dropna()
    df_2000w = df_2000w[~df_2000w['COUNTY/CITY'].str.contains('Total')].reset_index(drop = True)
    df_2000 = pd.melt(df_2000w
                        , id_vars = ['COUNTY/CITY']
                        , var_name = 'Date'
                        , value_name = 'Total'
                     )
    df_2000['Date'] = pd.to_datetime(df_2000['Date'])
    df_2000w = df_2000.pivot_table(index = ['COUNTY/CITY']
                                       , columns = 'Date'
                                       , values = 'Total').reset_index()
    df_2000w.columns = [re.sub(" 00:00:00", "", str(col)) for col in df_2000w.columns]

    # 2010-2020
    url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2010-2020-Internet-Version.xlsx"
    request = requests.get(url, headers = headers)
    request = request.content
    df_2010w = pd.read_excel(request, sheet_name = 2, skiprows = 1, engine='openpyxl', usecols = 'A:L')
    df_2010w = df_2010w.dropna()
    df_2010w = df_2010w[~df_2010w['COUNTY/CITY'].str.contains('Total')].reset_index(drop = True)
    df_2010 = pd.melt(df_2010w
                        , id_vars = ['COUNTY/CITY']
                        , var_name = 'Date'
                        , value_name = 'Total'
                     )
    df_2010['Date'] = pd.to_datetime(df_2010['Date'])
    df_2010w = df_2010.pivot_table(index = ['COUNTY/CITY']
                                       , columns = 'Date'
                                       , values = 'Total').reset_index()
    df_2010w.columns = [re.sub(" 00:00:00", "", str(col)) for col in df_2010w.columns]
    
    # 2020-2024
    url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2024_InternetVersion.xlsx"
    request = requests.get(url, headers = headers)
    request = request.content
    df_2020w = pd.read_excel(request, sheet_name = 2, skiprows = 2, engine='openpyxl', usecols = 'A:F')
    df_2020w = df_2020w.dropna()
    df_2020w = df_2020w[~df_2020w['COUNTY/CITY'].str.contains('Total')].reset_index(drop = True)
    df_2020 = pd.melt(df_2020w
                        , id_vars = ['COUNTY/CITY']
                        , var_name = 'Date'
                        , value_name = 'Total'
                     )
    df_2020['Date'] = pd.to_datetime(df_2020['Date'])
    df_2020w = df_2020.pivot_table(index = ['COUNTY/CITY']
                                       , columns = 'Date'
                                       , values = 'Total').reset_index()
    df_2020w.columns = [re.sub(" 00:00:00", "", str(col)) for col in df_2020w.columns]
    
    
    ## Combine data
    
    # wide
    df_dof2 = df_2000w.merge(df_2010w, on = 'COUNTY/CITY', how = 'left')
    df_dof2 = df_dof2 .merge(df_2020w, on = 'COUNTY/CITY', how = 'left')
    
    # long
    df_dof = pd.concat([df_2000, df_2010, df_2020])
    df_dof = df_dof.sort_values(['COUNTY/CITY', 'Date'])

    # Clean COUNTY field
    df_dof ['COUNTY/CITY'] = df_dof ['COUNTY/CITY'].apply(lambda s : re.sub('[\s+]', ' ', s.strip()))
    df_dof2['COUNTY/CITY'] = df_dof2['COUNTY/CITY'].apply(lambda s : re.sub('[\s+]', ' ', s.strip()))

print('Success!  DOF data imported correctly probably')

Success!  DOF data imported correctly probably


In [56]:
# Export to SP
set(df_dof['COUNTY/CITY'].values)
# Need to double check if:
# These are actually jurisdictions
# do we want "Incorporated" or other weird assignments

{'Adelanto',
 'Agoura Hills',
 'Alameda',
 'Albany',
 'Alhambra',
 'Aliso Viejo',
 'Alpine County',
 'Alturas',
 'Amador',
 'American Canyon',
 'Anaheim',
 'Anderson',
 'Angels City',
 'Antioch',
 'Apple Valley',
 'Arcadia',
 'Arcata',
 'Arroyo Grande',
 'Artesia',
 'Arvin',
 'Atascadero',
 'Atherton',
 'Atwater',
 'Auburn',
 'Avalon',
 'Avenal',
 'Azusa',
 'Bakersfield',
 'Balance Of County',
 'Baldwin Park',
 'Banning',
 'Barstow',
 'Beaumont',
 'Bell',
 'Bell Gardens',
 'Bellflower',
 'Belmont',
 'Belvedere',
 'Benicia',
 'Berkeley',
 'Beverly Hills',
 'Big Bear Lake',
 'Biggs',
 'Bishop',
 'Blue Lake',
 'Blythe',
 'Bradbury',
 'Brawley',
 'Brea',
 'Brentwood',
 'Brisbane',
 'Buellton',
 'Buena Park',
 'Burbank',
 'Burlingame',
 'Calabasas',
 'Calexico',
 'California City',
 'Calimesa',
 'Calipatria',
 'Calistoga',
 'Camarillo',
 'Campbell',
 'Canyon Lake',
 'Capitola',
 'Carlsbad',
 'Carmel-By-The-Sea',
 'Carpinteria',
 'Carson',
 'Cathedral City',
 'Ceres',
 'Cerritos',
 'Chico',


***

## Cost_3

***

Code graveyard

In [ ]:
# header = {
#   "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
#   "X-Requested-With": "XMLHttpRequest"
# }

# r = requests.get(site, headers=header)
# r.text

In [ ]:
# import pandas as pd
# import requests

# # Check the end of the url -->                                                                             HERE --v
# url = 'https://<myOrg>.sharepoint.com/:x:/s/x-taulukot/Ec0R1y3l7sdGsP92csSO-mgBI8WCN153LfEMvzKMSg1Zzg?e=6NS5Qh&download=1'
# headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/114.0'}

# resp = requests.get(url, headers=headers)
# df = pd.read_excel(resp.content, engine='openpyxl')